In [ ]:
from pathlib import Path


def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for path in (current, *current.parents):
        if (path / "data").exists():
            return path
    return current


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data"

import time
from datetime import datetime

import baostock as bs
import pandas as pd

OUT_DIR = DATA_ROOT / "data_download" / "tusahre_day_download" / "stock_day"
CACHE_DIR = OUT_DIR / "cache_batches"
START_DATE = "2015-01-01"
END_DATE = "2015-12-31"
FIELDS = "date,code,open,high,low,close,preclose,volume,amount,pctChg,turn,tradestatus,isST"
COLUMNS = FIELDS.split(",")
ADJUSTFLAG = "1"
BATCH_SIZE = 400
OVERWRITE = False

OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def login(max_tries=5, sleep_sec=2):
    last_error = "baostock login failed"
    for _ in range(max_tries):
        result = bs.login()
        if result.error_code == "0":
            return
        last_error = result.error_msg
        try:
            bs.logout()
        except Exception:
            pass
        time.sleep(sleep_sec)
    raise RuntimeError(last_error)


def result_to_frame(result, columns):
    rows = []
    while result.error_code == "0" and result.next():
        rows.append(result.get_row_data())
    return pd.DataFrame(rows, columns=columns)


def normalize_daily_frame(df):
    if df.empty:
        return df.reindex(columns=COLUMNS)
    df = df.reindex(columns=COLUMNS)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    df["code"] = df["code"].astype(str).str.split(".").str[-1].str.zfill(6)
    for col in ["open", "high", "low", "close", "preclose", "volume", "amount", "pctChg", "turn"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")
    for col in ["tradestatus", "isST"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int8")
    df["date"] = df["date"].dt.strftime("%Y%m%d").astype(int)
    return df[COLUMNS]


def fetch_stock(code, start_date, end_date, retries=3):
    for attempt in range(retries):
        result = bs.query_history_k_data_plus(
            code,
            FIELDS,
            start_date=start_date,
            end_date=end_date,
            frequency="d",
            adjustflag=ADJUSTFLAG,
        )
        if result.error_code == "0":
            return normalize_daily_frame(result_to_frame(result, COLUMNS))
        time.sleep(0.3 * (attempt + 1))
    return pd.DataFrame(columns=COLUMNS)


def get_stock_codes(start_date, end_date):
    result = bs.query_stock_basic()
    basic = result_to_frame(result, ["code", "ipoDate", "outDate", "type", "status"])
    if basic.empty:
        return []
    basic = basic[basic["type"] == "1"].copy()
    basic["ipoDate"] = pd.to_datetime(basic["ipoDate"], errors="coerce")
    basic["outDate"] = pd.to_datetime(basic["outDate"].replace("", pd.NA), errors="coerce")
    start_ts = pd.to_datetime(start_date)
    end_ts = min(pd.to_datetime(end_date), pd.Timestamp.today().normalize())
    mask = basic["ipoDate"].le(end_ts) & (basic["outDate"].isna() | basic["outDate"].ge(start_ts))
    return basic.loc[mask, "code"].dropna().astype(str).sort_values().tolist()


def write_daily_files(frame):
    if frame.empty:
        return 0
    written = 0
    for day, day_df in frame.groupby("date", sort=True):
        out_path = OUT_DIR / f"{int(day)}.parquet"
        if out_path.exists() and not OVERWRITE:
            continue
        day_df.sort_values("code").to_parquet(out_path, index=False)
        written += 1
    return written


def main():
    login()
    try:
        codes = get_stock_codes(START_DATE, END_DATE)
        total_written = 0
        for batch_id, start in enumerate(range(0, len(codes), BATCH_SIZE), start=1):
            batch_codes = codes[start:start + BATCH_SIZE]
            cache_path = CACHE_DIR / f"batch_{batch_id:05d}.parquet"
            if cache_path.exists() and not OVERWRITE:
                batch_df = pd.read_parquet(cache_path)
            else:
                frames = [fetch_stock(code, START_DATE, END_DATE) for code in batch_codes]
                batch_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=COLUMNS)
                batch_df.to_parquet(cache_path, index=False)
            total_written += write_daily_files(batch_df)
            print(f"batch={batch_id} stocks={len(batch_codes)} rows={len(batch_df)}")
        print(f"done files={total_written} output={OUT_DIR}")
    finally:
        bs.logout()


if __name__ == "__main__":
    main()